#Q3 Fine-tuning a Transformer-based Model

In [2]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


##DeBERTA Transformers Model

In [ ]:
!pip install transformers datasets scikit-learn accelerate -q

import os
import csv
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from transformers import AutoModel, AutoTokenizer, Trainer, TrainingArguments
from datasets import Dataset
from sklearn.metrics import mean_absolute_error, accuracy_score, precision_score, recall_score, f1_score, classification_report

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    torch.cuda.empty_cache()

# ============================================================================
# DATA LOADING
# ============================================================================

print("\n" + "="*80)
print("DATA LOADING")
print("="*80)

DATA_DIR = "/content/drive/MyDrive/P1_data"
TRAIN_CSV = os.path.join(DATA_DIR, "trac2_CONVT_train.csv")
DEV_CSV   = os.path.join(DATA_DIR, "trac2_CONVT_dev.csv")
TEST_CSV  = os.path.join(DATA_DIR, "trac2_CONVT_test.csv")

def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    df.columns = [str(c).strip().strip('"') for c in df.columns]
    return df

def safe_read_csv(path):

    df = pd.read_csv(
        path,
        engine="python",
        quoting=csv.QUOTE_MINIMAL,
        quotechar='"',
        escapechar='\\',
        encoding='utf-8',
        on_bad_lines='skip'
    )
    return normalize_columns(df)

train_df = safe_read_csv(TRAIN_CSV)
dev_df   = safe_read_csv(DEV_CSV)
test_raw = safe_read_csv(TEST_CSV)

# Clean column names
train_df = normalize_columns(train_df)
dev_df   = normalize_columns(dev_df)
test_raw = normalize_columns(test_raw)

print("Detected columns:")
print("  Train:", train_df.columns.tolist())
print("  Dev:  ", dev_df.columns.tolist())
print("  Test: ", test_raw.columns.tolist())

print(f" Raw shapes -> Train: {train_df.shape}, Dev: {dev_df.shape}, Test: {test_raw.shape}")

# ============================================================================
# DATA PREPROCESSING
# ============================================================================

print("\n" + "="*80)
print("DATA PREPROCESSING")
print("="*80)

# Keep only needed columns for train/dev
required_train_cols = {"id", "text", "Emotion", "EmotionalPolarity", "Empathy"}
if not required_train_cols.issubset(set(train_df.columns)):
    missing = required_train_cols - set(train_df.columns)
    raise ValueError(f"Train CSV missing columns: {missing}")
if not required_train_cols.issubset(set(dev_df.columns)):
    missing = required_train_cols - set(dev_df.columns)
    raise ValueError(f"Dev CSV missing columns: {missing}")

train_df = train_df[["id", "text", "Emotion", "EmotionalPolarity", "Empathy"]].copy()
dev_df   = dev_df[  ["id", "text", "Emotion", "EmotionalPolarity", "Empathy"]].copy()

# Select the correct test columns explicitly
if not {"id", "text"}.issubset(set(test_raw.columns)):
    missing = {"id", "text"} - set(test_raw.columns)
    raise ValueError(f"Test CSV missing columns: {missing}")

test_df = test_raw[["id", "text"]].copy()

# Type enforcement
for df_ in (train_df, dev_df, test_df):
    df_["text"] = df_["text"].astype(str)
train_df["id"] = pd.to_numeric(train_df["id"], errors="coerce")
dev_df["id"]   = pd.to_numeric(dev_df["id"], errors="coerce")
test_df["id"]  = pd.to_numeric(test_df["id"], errors="coerce")

# Drop NaNs in labels for train/dev
train_df = train_df.dropna(subset=['Emotion', 'EmotionalPolarity', 'Empathy'])
dev_df   = dev_df.dropna(subset=['Emotion', 'EmotionalPolarity', 'Empathy'])

# Handle polarity - clip to [1,3] then convert to [0,1,2]
train_df['EmotionalPolarity'] = train_df['EmotionalPolarity'].clip(1, 3).astype(int) - 1
dev_df['EmotionalPolarity']   = dev_df['EmotionalPolarity'].clip(1, 3).astype(int) - 1

print(f" After preprocessing:")
print(f"  Train: {train_df.shape}")
print(f"  Dev:   {dev_df.shape}")
print(f"  Test:  {test_df.shape}")
print(f"\n Polarity distribution:")
print(f"  Train: {train_df['EmotionalPolarity'].value_counts().sort_index().to_dict()}")
print(f"  Dev:   {dev_df['EmotionalPolarity'].value_counts().sort_index().to_dict()}")

NUM_LABELS = 3

# ============================================================================
# MODEL CONFIGURATION
# ============================================================================

print("\n" + "="*80)
print("MODEL CONFIGURATION")
print("="*80)

MODEL_NAME = "microsoft/deberta-v3-base"
MAX_LEN = 128

LEARNING_RATE = 2.0e-05
BATCH_SIZE = 16
EPOCHS = 3
WEIGHT_DECAY = 0.015
WARMUP_RATIO = 0.06
DROPOUT = 0.12

print(f"Model: {MODEL_NAME}")
print(f"Learning Rate: {LEARNING_RATE}")
print(f"Batch Size: {BATCH_SIZE}")
print(f"Epochs: {EPOCHS}")
print(f"Dropout: {DROPOUT:.4f}")
print(f"Weight Decay: {WEIGHT_DECAY:.4f}")
print(f"Warmup Ratio: {WARMUP_RATIO:.4f}")
print(f"Polarity Classes: {NUM_LABELS}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"\n✓ Tokenizer loaded: {tokenizer.__class__.__name__}")

# ============================================================================
# TOKENIZATION
# ============================================================================

print("\n" + "="*80)
print("TOKENIZATION")
print("="*80)

# Rename label columns
train_df = train_df.rename(columns={'Emotion': 'emotion', 'EmotionalPolarity': 'polarity', 'Empathy': 'empathy'})
dev_df   = dev_df.rename(columns={'Emotion': 'emotion', 'EmotionalPolarity': 'polarity', 'Empathy': 'empathy'})

# Create datasets
train_ds = Dataset.from_pandas(train_df[['text', 'emotion', 'polarity', 'empathy']])
dev_ds   = Dataset.from_pandas(dev_df[['text', 'emotion', 'polarity', 'empathy']])
test_ds  = Dataset.from_pandas(test_df[['text']])

def tokenize_function(examples):
    return tokenizer(examples['text'], truncation=True, padding=False, max_length=MAX_LEN)

train_ds = train_ds.map(tokenize_function, batched=True)
dev_ds   = dev_ds.map(tokenize_function, batched=True)
test_ds  = test_ds.map(tokenize_function, batched=True)

print(f" Train: {len(train_ds)} samples")
print(f" Dev:   {len(dev_ds)} samples")
print(f" Test:  {len(test_ds)} samples")

# ============================================================================
# MODEL DEFINITION
# ============================================================================

class MultiTaskBERT(nn.Module):
    def __init__(self, model_name, num_polarity_classes, dropout=0.1):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size

        self.dropout = nn.Dropout(dropout)
        self.emotion_head  = nn.Linear(hidden_size, 1)
        self.empathy_head  = nn.Linear(hidden_size, 1)
        self.polarity_head = nn.Linear(hidden_size, num_polarity_classes)

    def forward(self, input_ids, attention_mask, emotion=None, empathy=None, polarity=None):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]
        cls_output = self.dropout(cls_output)

        emotion_pred   = self.emotion_head(cls_output).squeeze(-1)
        empathy_pred   = self.empathy_head(cls_output).squeeze(-1)
        polarity_logits= self.polarity_head(cls_output)

        loss = None
        if emotion is not None and empathy is not None and polarity is not None:
            mse_loss = nn.MSELoss()
            ce_loss  = nn.CrossEntropyLoss()
            loss_emotion  = mse_loss(emotion_pred, emotion)
            loss_empathy  = mse_loss(empathy_pred, empathy)
            loss_polarity = ce_loss(polarity_logits, polarity)
            loss = loss_emotion + loss_empathy + loss_polarity

        return {
            'loss': loss,
            'emotion': emotion_pred,
            'empathy': empathy_pred,
            'polarity_logits': polarity_logits
        }

# ============================================================================
# CUSTOM TRAINER
# ============================================================================

class MultiTaskTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = {
            'emotion': inputs.pop('emotion'),
            'empathy': inputs.pop('empathy'),
            'polarity': inputs.pop('polarity')
        }

        outputs = model(
            input_ids=inputs['input_ids'],
            attention_mask=inputs['attention_mask'],
            emotion=labels['emotion'],
            empathy=labels['empathy'],
            polarity=labels['polarity']
        )

        loss = outputs['loss']
        return (loss, outputs) if return_outputs else loss

    def prediction_step(self, model, inputs, prediction_loss_only, ignore_keys=None):
        has_labels = 'emotion' in inputs

        if has_labels:
            labels = {
                'emotion': inputs.pop('emotion'),
                'empathy': inputs.pop('empathy'),
                'polarity': inputs.pop('polarity')
            }

        with torch.no_grad():
            outputs = model(
                input_ids=inputs['input_ids'],
                attention_mask=inputs['attention_mask'],
                emotion=labels['emotion'] if has_labels else None,
                empathy=labels['empathy'] if has_labels else None,
                polarity=labels['polarity'] if has_labels else None
            )

        loss = outputs['loss'] if has_labels else None

        predictions = (
            outputs['emotion'].detach().cpu(),
            outputs['empathy'].detach().cpu(),
            outputs['polarity_logits'].detach().cpu()
        )

        if has_labels:
            labels_tuple = (
                labels['emotion'].detach().cpu(),
                labels['empathy'].detach().cpu(),
                labels['polarity'].detach().cpu()
            )
        else:
            labels_tuple = None

        return (loss, predictions, labels_tuple)

# ============================================================================
# DATA COLLATOR
# ============================================================================

from dataclasses import dataclass
from typing import Dict, List

@dataclass
class MultiTaskCollator:
    tokenizer: AutoTokenizer

    def __call__(self, features: List[Dict]) -> Dict[str, torch.Tensor]:
        input_ids = [torch.tensor(f['input_ids']) for f in features]
        attention_mask = [torch.tensor(f['attention_mask']) for f in features]

        max_len = max(len(ids) for ids in input_ids)

        padded_input_ids = []
        padded_attention_mask = []

        for ids, mask in zip(input_ids, attention_mask):
            padding_length = max_len - len(ids)
            padded_input_ids.append(torch.cat([ids, torch.zeros(padding_length, dtype=torch.long)]))
            padded_attention_mask.append(torch.cat([mask, torch.zeros(padding_length, dtype=torch.long)]))

        batch = {
            'input_ids': torch.stack(padded_input_ids),
            'attention_mask': torch.stack(padded_attention_mask)
        }

        if 'emotion' in features[0]:
            batch['emotion'] = torch.tensor([f['emotion'] for f in features], dtype=torch.float)
            batch['empathy'] = torch.tensor([f['empathy'] for f in features], dtype=torch.float)
            batch['polarity'] = torch.tensor([f['polarity'] for f in features], dtype=torch.long)

        return batch

# ============================================================================
# COMPUTE METRICS
# ============================================================================

def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    pred_emotion = predictions[0]
    pred_empathy = predictions[1]
    pred_polarity_logits = predictions[2]

    label_emotion = labels[0]
    label_empathy = labels[1]
    label_polarity = labels[2]

    mae_emotion = mean_absolute_error(label_emotion, pred_emotion)
    mae_empathy = mean_absolute_error(label_empathy, pred_empathy)

    pred_polarity_class = np.argmax(pred_polarity_logits, axis=1)
    acc_polarity = accuracy_score(label_polarity, pred_polarity_class)
    f1_polarity = f1_score(label_polarity, pred_polarity_class, average='weighted', zero_division=0)

    return {
        'mae_emotion': mae_emotion,
        'mae_empathy': mae_empathy,
        'accuracy_polarity': acc_polarity,
        'f1_polarity': f1_polarity
    }


PyTorch: 2.8.0+cu126
CUDA: True
Device: Tesla T4

DATA LOADING
Detected columns:
  Train: ['id', 'article_id', 'conversation_id', 'turn_id', 'speaker', 'text', 'person_id_1', 'person_id_2', 'Emotion', 'EmotionalPolarity', 'Empathy', 'SelfDisclosure']
  Dev:   ['id', 'article_id', 'conversation_id', 'turn_id', 'speaker_id', 'text', 'person_id', 'person_id_1', 'person_id_2', 'Emotion', 'EmotionalPolarity', 'Empathy', 'SelfDisclosure']
  Test:  ['id', 'article_id', 'conversation_id', 'turn_id', 'speaker_id', 'text', 'person_id', 'person_id_1', 'person_id_2']
 Raw shapes -> Train: (10941, 12), Dev: (990, 13), Test: (2316, 9)

DATA PREPROCESSING
 After preprocessing:
  Train: (10941, 5)
  Dev:   (990, 5)
  Test:  (2316, 2)

 Polarity distribution:
  Train: {0: 6263, 1: 4677, 2: 1}
  Dev:   {0: 631, 1: 359}

MODEL CONFIGURATION
Model: microsoft/deberta-v3-base
Learning Rate: 2e-05
Batch Size: 16
Epochs: 3
Dropout: 0.1200
Weight Decay: 0.0150
Warmup Ratio: 0.0600
Polarity Classes: 3


/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(



✓ Tokenizer loaded: DebertaV2TokenizerFast

TOKENIZATION


Map:   0%|          | 0/10941 [00:00<?, ? examples/s]

Map:   0%|          | 0/990 [00:00<?, ? examples/s]

Map:   0%|          | 0/2316 [00:00<?, ? examples/s]

 Train: 10941 samples
 Dev:   990 samples
 Test:  2316 samples


In [ ]:
# ============================================================================
# TRAINING
# ============================================================================

print("\n" + "="*80)
print("TRAINING")
print("="*80)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

collator = MultiTaskCollator(tokenizer)

model = MultiTaskBERT(MODEL_NAME, NUM_LABELS, dropout=DROPOUT).to(device)
print(f" Model initialized with {sum(p.numel() for p in model.parameters()):,} parameters")

training_args = TrainingArguments(
    output_dir="./outputs_final",
    eval_strategy="epoch",
    save_strategy="epoch",
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    learning_rate=LEARNING_RATE,
    num_train_epochs=EPOCHS,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    logging_steps=50,
    fp16=torch.cuda.is_available(),
    load_best_model_at_end=True,
    metric_for_best_model="eval_accuracy_polarity",
    greater_is_better=True,
    report_to="none",
    save_total_limit=2
)

trainer = MultiTaskTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=dev_ds,
    data_collator=collator,
    compute_metrics=compute_metrics
)

print("\nStarting training...")
trainer.train()
print("\n✓ Training complete!")

# ============================================================================
# DEV SET EVALUATION
# ============================================================================

print("\n" + "="*80)
print("DEV SET EVALUATION")
print("="*80)

dev_predictions = trainer.predict(dev_ds)

pred_dev_emotion = dev_predictions.predictions[0]
pred_dev_empathy = dev_predictions.predictions[1]
pred_dev_polarity_logits = dev_predictions.predictions[2]
pred_dev_polarity = np.argmax(pred_dev_polarity_logits, axis=1)

true_dev_emotion = dev_predictions.label_ids[0]
true_dev_empathy = dev_predictions.label_ids[1]
true_dev_polarity = dev_predictions.label_ids[2]

# 1. EMOTION (Regression)
mae_emotion = mean_absolute_error(true_dev_emotion, pred_dev_emotion)
print("\n1. EMOTION (Regression Task)")
print("-" * 60)
print(f"   Mean Absolute Error (MAE): {mae_emotion:.4f}")

# 2. EMPATHY (Regression)
mae_empathy = mean_absolute_error(true_dev_empathy, pred_dev_empathy)
print("\n2. EMPATHY (Regression Task)")
print("-" * 60)
print(f"   Mean Absolute Error (MAE): {mae_empathy:.4f}")

# 3. EMOTIONAL POLARITY (Classification)
acc_polarity = accuracy_score(true_dev_polarity, pred_dev_polarity)
prec_polarity = precision_score(true_dev_polarity, pred_dev_polarity, average='weighted', zero_division=0)
rec_polarity = recall_score(true_dev_polarity, pred_dev_polarity, average='weighted', zero_division=0)
f1_polarity = f1_score(true_dev_polarity, pred_dev_polarity, average='weighted', zero_division=0)

print("\n3. EMOTIONAL POLARITY (Classification Task)")
print("-" * 60)
print(f"   Accuracy:  {acc_polarity:.4f}")
print(f"   Precision: {prec_polarity:.4f}")
print(f"   Recall:    {rec_polarity:.4f}")
print(f"   F1-Score:  {f1_polarity:.4f}")

# Detailed classification report
unique_classes = sorted(set(true_dev_polarity))
target_names = [f'Class {i}' for i in unique_classes]
print("\n   Classification Report:")
print(classification_report(true_dev_polarity, pred_dev_polarity,
                          labels=unique_classes, target_names=target_names))

# Save dev set predictions to CSV
# Convert polarity back to original range [1, 2, 3]
pred_dev_polarity_original = pred_dev_polarity + 1
true_dev_polarity_original = true_dev_polarity + 1

dev_results_df = pd.DataFrame({
    'id': dev_df['id'].values,
    'true_Emotion': true_dev_emotion,
    'pred_Emotion': pred_dev_emotion,
    'true_EmotionalPolarity': true_dev_polarity_original,
    'pred_EmotionalPolarity': pred_dev_polarity_original,
    'true_Empathy': true_dev_empathy,
    'pred_Empathy': pred_dev_empathy
})

dev_results_df.to_csv('dev_predictions_bert.csv', index=False)
print(f"\n Dev predictions saved to: dev_predictions_bert.csv")
print(f" Dev predictions shape: {dev_results_df.shape}")

# Save model
trainer.save_model("./best_model_deberta")
tokenizer.save_pretrained("./best_model_deberta")
print("\n Model saved to ./best_model_deberta")

# ============================================================================
# TEST SET PREDICTIONS
# ============================================================================

print("\n" + "="*80)
print("TEST SET PREDICTIONS")
print("="*80)

test_predictions = trainer.predict(test_ds)

pred_test_emotion = test_predictions.predictions[0]
pred_test_empathy = test_predictions.predictions[1]
pred_test_polarity_logits = test_predictions.predictions[2]
pred_test_polarity = np.argmax(pred_test_polarity_logits, axis=1)

# Convert polarity back to original range [1, 2, 3]
pred_test_polarity_original = pred_test_polarity + 1

# Create submission CSV
submission_df = pd.DataFrame({
    'id': test_df['id'].values,
    'Emotion': pred_test_emotion,
    'EmotionalPolarity': pred_test_polarity_original,
    'Empathy': pred_test_empathy
})

submission_df.to_csv('predictions_bert.csv', index=False)

print(f"\n Predictions saved to: predictions_bert.csv")
print(f" Submission shape: {submission_df.shape}")
print(f" Columns: {submission_df.columns.tolist()}")

print("\nSummary statistics:")
print(submission_df.describe())

# ============================================================================
# FINAL SUMMARY
# ============================================================================

print("\n" + "="*80)
print("FINAL SUMMARY")
print("="*80)

print(f"\n Model: {MODEL_NAME}")
print(f"\n Dev Set Performance:")
print(f"   Emotion MAE:        {mae_emotion:.4f}")
print(f"   Empathy MAE:        {mae_empathy:.4f}")
print(f"   Polarity Accuracy:  {acc_polarity:.4f} ({acc_polarity*100:.2f}%)")
print(f"   Polarity Precision: {prec_polarity:.4f}")
print(f"   Polarity Recall:    {rec_polarity:.4f}")
print(f"   Polarity F1:        {f1_polarity:.4f}")

print(f"\n Output Files:")
print(f"   Model: ./best_model_deberta/")
print(f"   Dev Predictions: dev_predictions_bert.csv")
print(f"   Test Predictions: predictions_bert.csv")

print("\n" + "="*80)
print("ALL TASKS COMPLETED SUCCESSFULLY!")
print("="*80)


TRAINING
Device: cuda
 Model initialized with 183,835,397 parameters

Starting training...


Epoch,Training Loss,Validation Loss,Mae Emotion,Mae Empathy,Accuracy Polarity,F1 Polarity
1,1.403500,1.529187,0.502877,0.686918,0.837374,0.838308
2,1.177500,1.449998,0.462512,0.677906,0.838384,0.832364
3,1.060700,1.503456,0.486325,0.675719,0.825253,0.822119



✓ Training complete!

DEV SET EVALUATION



1. EMOTION (Regression Task)
------------------------------------------------------------
   Mean Absolute Error (MAE): 0.4625

2. EMPATHY (Regression Task)
------------------------------------------------------------
   Mean Absolute Error (MAE): 0.6779

3. EMOTIONAL POLARITY (Classification Task)
------------------------------------------------------------
   Accuracy:  0.8384
   Precision: 0.8419
   Recall:    0.8384
   F1-Score:  0.8324

   Classification Report:
              precision    recall  f1-score   support

     Class 0       0.83      0.94      0.88       631
     Class 1       0.87      0.65      0.75       359

    accuracy                           0.84       990
   macro avg       0.85      0.80      0.81       990
weighted avg       0.84      0.84      0.83       990


 Dev predictions saved to: dev_predictions_bert.csv
 Dev predictions shape: (990, 7)

 Model saved to ./best_model_deberta

TEST SET PREDICTIONS



 Predictions saved to: predictions_bert.csv
 Submission shape: (2316, 4)
 Columns: ['id', 'Emotion', 'EmotionalPolarity', 'Empathy']

Summary statistics:
                id      Emotion  EmotionalPolarity      Empathy
count  2316.000000  2316.000000        2316.000000  2316.000000
mean   1292.481002     2.093797           1.246978     2.302747
std     805.998458     0.736905           0.431347     0.840968
min       1.000000     0.866699           1.000000     0.899902
25%     579.750000     1.531250           1.000000     1.524658
50%    1252.500000     2.088867           1.000000     2.426758
75%    1966.250000     2.595703           1.000000     2.943359
max    2780.000000     4.253906           2.000000     4.046875

FINAL SUMMARY

 Model: microsoft/deberta-v3-base

 Dev Set Performance:
   Emotion MAE:        0.4625
   Empathy MAE:        0.6779
   Polarity Accuracy:  0.8384 (83.84%)
   Polarity Precision: 0.8419
   Polarity Recall:    0.8384
   Polarity F1:        0.8324

 Outpu

##Hyper parameter Tuning

In [ ]:
!pip install transformers datasets scikit-learn accelerate optuna -q

import os
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from transformers import AutoModel, AutoTokenizer, Trainer, TrainingArguments
from datasets import Dataset
from sklearn.metrics import mean_absolute_error, accuracy_score, f1_score, classification_report

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    torch.cuda.empty_cache()

# DATA LOADING & PREPROCESSING

print("\n" + "="*80)
print("DATA LOADING & PREPROCESSING")
print("="*80)

DATA_DIR = "/content/drive/MyDrive/P1_data"
TRAIN_CSV = os.path.join(DATA_DIR, "trac2_CONVT_train.csv")
DEV_CSV = os.path.join(DATA_DIR, "trac2_CONVT_dev.csv")
TEST_CSV = os.path.join(DATA_DIR, "trac2_CONVT_test.csv")

def safe_read_csv(path):
    try:
        df = pd.read_csv(path, quoting=3, on_bad_lines='skip', engine='python', encoding='utf-8')
    except Exception as e:
        df = pd.read_csv(path, sep=',', engine='python', on_bad_lines='skip', encoding='utf-8')
    return df

train_df = safe_read_csv(TRAIN_CSV)
dev_df = safe_read_csv(DEV_CSV)
test_df = safe_read_csv(TEST_CSV)

# Clean column names
train_df.columns = train_df.columns.str.strip('"')
dev_df.columns = dev_df.columns.str.strip('"')
test_df.columns = test_df.columns.str.strip('"')

# Keep only needed columns
train_df = train_df[["id", "text", "Emotion", "EmotionalPolarity", "Empathy"]]
dev_df = dev_df[["id", "text", "Emotion", "EmotionalPolarity", "Empathy"]]
test_df = test_df[["id", "text"]]

# Drop NaNs
train_df = train_df.dropna(subset=['Emotion', 'EmotionalPolarity', 'Empathy'])
dev_df = dev_df.dropna(subset=['Emotion', 'EmotionalPolarity', 'Empathy'])

print(f"✓ Train: {train_df.shape}, Dev: {dev_df.shape}, Test: {test_df.shape}")

# Handle polarity
train_df['EmotionalPolarity'] = train_df['EmotionalPolarity'].clip(1, 3).astype(int) - 1
dev_df['EmotionalPolarity'] = dev_df['EmotionalPolarity'].clip(1, 3).astype(int) - 1

print(f"✓ Polarity: Train {train_df['EmotionalPolarity'].value_counts().sort_index().to_dict()}")
print(f"✓ Polarity: Dev {dev_df['EmotionalPolarity'].value_counts().sort_index().to_dict()}")

NUM_LABELS = 3

# MODEL & TOKENIZER

print("\n" + "="*80)
print("MODEL SETUP")
print("="*80)

MODEL_NAME = "microsoft/deberta-v3-base"
MAX_LEN = 128

print(f"Model: {MODEL_NAME}")
print(f"Max length: {MAX_LEN}")
print(f"Num polarity classes: {NUM_LABELS}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"✓ Tokenizer loaded: {tokenizer.__class__.__name__}")

# TOKENIZATION

print("\n" + "="*80)
print("TOKENIZATION")
print("="*80)

# Rename columns to lowercase
train_df = train_df.rename(columns={'Emotion': 'emotion', 'EmotionalPolarity': 'polarity', 'Empathy': 'empathy'})
dev_df = dev_df.rename(columns={'Emotion': 'emotion', 'EmotionalPolarity': 'polarity', 'Empathy': 'empathy'})

train_ds = Dataset.from_pandas(train_df[['text', 'emotion', 'polarity', 'empathy']])
dev_ds = Dataset.from_pandas(dev_df[['text', 'emotion', 'polarity', 'empathy']])
test_ds = Dataset.from_pandas(test_df[['text']])

def tokenize_function(examples):
    return tokenizer(examples['text'], truncation=True, padding=False, max_length=MAX_LEN)

train_ds = train_ds.map(tokenize_function, batched=True)
dev_ds = dev_ds.map(tokenize_function, batched=True)
test_ds = test_ds.map(tokenize_function, batched=True)

print(f"✓ Train: {len(train_ds)}, Dev: {len(dev_ds)}, Test: {len(test_ds)}")
print(f"✓ Columns: {train_ds.column_names}")

# MULTI-TASK MODEL

class MultiTaskBERT(nn.Module):
    def __init__(self, model_name, num_polarity_classes, dropout=0.1):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size

        self.dropout = nn.Dropout(dropout)
        self.emotion_head = nn.Linear(hidden_size, 1)
        self.empathy_head = nn.Linear(hidden_size, 1)
        self.polarity_head = nn.Linear(hidden_size, num_polarity_classes)

    def forward(self, input_ids, attention_mask, emotion=None, empathy=None, polarity=None):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]
        cls_output = self.dropout(cls_output)

        emotion_pred = self.emotion_head(cls_output).squeeze(-1)
        empathy_pred = self.empathy_head(cls_output).squeeze(-1)
        polarity_logits = self.polarity_head(cls_output)

        loss = None
        if emotion is not None and empathy is not None and polarity is not None:
            mse_loss = nn.MSELoss()
            ce_loss = nn.CrossEntropyLoss()

            loss_emotion = mse_loss(emotion_pred, emotion)
            loss_empathy = mse_loss(empathy_pred, empathy)
            loss_polarity = ce_loss(polarity_logits, polarity)

            loss = loss_emotion + loss_empathy + loss_polarity

        return {
            'loss': loss,
            'emotion': emotion_pred,
            'empathy': empathy_pred,
            'polarity_logits': polarity_logits
        }

# CUSTOM TRAINER

class MultiTaskTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = {
            'emotion': inputs.pop('emotion'),
            'empathy': inputs.pop('empathy'),
            'polarity': inputs.pop('polarity')
        }

        outputs = model(
            input_ids=inputs['input_ids'],
            attention_mask=inputs['attention_mask'],
            emotion=labels['emotion'],
            empathy=labels['empathy'],
            polarity=labels['polarity']
        )

        loss = outputs['loss']
        return (loss, outputs) if return_outputs else loss

    def prediction_step(self, model, inputs, prediction_loss_only, ignore_keys=None):
        has_labels = 'emotion' in inputs

        if has_labels:
            labels = {
                'emotion': inputs.pop('emotion'),
                'empathy': inputs.pop('empathy'),
                'polarity': inputs.pop('polarity')
            }

        with torch.no_grad():
            outputs = model(
                input_ids=inputs['input_ids'],
                attention_mask=inputs['attention_mask'],
                emotion=labels['emotion'] if has_labels else None,
                empathy=labels['empathy'] if has_labels else None,
                polarity=labels['polarity'] if has_labels else None
            )

        loss = outputs['loss'] if has_labels else None

        predictions = (
            outputs['emotion'].detach().cpu(),
            outputs['empathy'].detach().cpu(),
            outputs['polarity_logits'].detach().cpu()
        )

        if has_labels:
            labels_tuple = (
                labels['emotion'].detach().cpu(),
                labels['empathy'].detach().cpu(),
                labels['polarity'].detach().cpu()
            )
        else:
            labels_tuple = None

        return (loss, predictions, labels_tuple)

# DATA COLLATOR

from dataclasses import dataclass
from typing import Dict, List

@dataclass
class MultiTaskCollator:
    tokenizer: AutoTokenizer

    def __call__(self, features: List[Dict]) -> Dict[str, torch.Tensor]:
        input_ids = [torch.tensor(f['input_ids']) for f in features]
        attention_mask = [torch.tensor(f['attention_mask']) for f in features]

        max_len = max(len(ids) for ids in input_ids)

        padded_input_ids = []
        padded_attention_mask = []

        for ids, mask in zip(input_ids, attention_mask):
            padding_length = max_len - len(ids)
            padded_input_ids.append(torch.cat([ids, torch.zeros(padding_length, dtype=torch.long)]))
            padded_attention_mask.append(torch.cat([mask, torch.zeros(padding_length, dtype=torch.long)]))

        batch = {
            'input_ids': torch.stack(padded_input_ids),
            'attention_mask': torch.stack(padded_attention_mask)
        }

        if 'emotion' in features[0]:
            batch['emotion'] = torch.tensor([f['emotion'] for f in features], dtype=torch.float)
            batch['empathy'] = torch.tensor([f['empathy'] for f in features], dtype=torch.float)
            batch['polarity'] = torch.tensor([f['polarity'] for f in features], dtype=torch.long)

        return batch

# COMPUTE METRICS

def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    pred_emotion = predictions[0]
    pred_empathy = predictions[1]
    pred_polarity_logits = predictions[2]

    label_emotion = labels[0]
    label_empathy = labels[1]
    label_polarity = labels[2]

    mae_emotion = mean_absolute_error(label_emotion, pred_emotion)
    mae_empathy = mean_absolute_error(label_empathy, pred_empathy)

    pred_polarity_class = np.argmax(pred_polarity_logits, axis=1)
    acc_polarity = accuracy_score(label_polarity, pred_polarity_class)
    f1_polarity = f1_score(label_polarity, pred_polarity_class, average='weighted', zero_division=0)

    return {
        'mae_emotion': mae_emotion,
        'mae_empathy': mae_empathy,
        'accuracy_polarity': acc_polarity,
        'f1_polarity': f1_polarity
    }

# MODEL INITIALIZATION FUNCTION FOR HP SEARCH

def model_init(trial):
    """Model initialization function for hyperparameter search"""
    # Get dropout from trial or use default
    dropout = trial.suggest_float("dropout", 0.1, 0.3) if trial is not None else 0.1
    return MultiTaskBERT(MODEL_NAME, NUM_LABELS, dropout=dropout)

# HYPERPARAMETER SEARCH SPACE

def optuna_hp_space(trial):
    """Define hyperparameter search space for Optuna"""
    return {
        "learning_rate": trial.suggest_float("learning_rate", 1e-5, 5e-5, log=True),
        "per_device_train_batch_size": trial.suggest_categorical("per_device_train_batch_size", [8, 16]),
        "num_train_epochs": trial.suggest_int("num_train_epochs", 3, 5),
        "weight_decay": trial.suggest_float("weight_decay", 0.001, 0.05),
        "warmup_ratio": trial.suggest_float("warmup_ratio", 0.05, 0.15),
    }

# TRAINING ARGUMENTS

print("\n" + "="*80)
print("HYPERPARAMETER SEARCH WITH TRAINER API")
print("="*80)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

collator = MultiTaskCollator(tokenizer)

# Base training arguments
training_args = TrainingArguments(
    output_dir="./outputs_hp_search",
    eval_strategy="epoch",
    save_strategy="epoch",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    num_train_epochs=4,
    weight_decay=0.01,
    warmup_ratio=0.1,
    logging_steps=100,
    fp16=torch.cuda.is_available(),
    load_best_model_at_end=True,
    metric_for_best_model="eval_accuracy_polarity",
    greater_is_better=True,
    report_to="none",
    save_total_limit=1,
)

# Create trainer
trainer = MultiTaskTrainer(
    model=None,
    model_init=model_init,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=dev_ds,
    data_collator=collator,
    compute_metrics=compute_metrics
)

# RUN HYPERPARAMETER SEARCH

print("\nStarting hyperparameter search ")
print("="*80)

best_trial = trainer.hyperparameter_search(
    direction="maximize",
    backend="optuna",
    hp_space=optuna_hp_space,
    n_trials=10,
    compute_objective=lambda metrics: metrics["eval_accuracy_polarity"]
)

print("\n" + "="*80)
print("HYPERPARAMETER SEARCH COMPLETE")
print("="*80)
print("\n Best hyperparameters found:")
for key, value in best_trial.hyperparameters.items():
    print(f"  {key}: {value}")
print(f"\n Best accuracy: {best_trial.objective:.4f}")


# TRAIN FINAL MODEL WITH BEST HYPERPARAMETERS

print("\n" + "="*80)
print("TRAINING FINAL MODEL WITH BEST HYPERPARAMETERS")
print("="*80)

# Update training arguments with best hyperparameters
for key, value in best_trial.hyperparameters.items():
    setattr(training_args, key, value)

training_args.output_dir = "./outputs_final"
training_args.num_train_epochs = best_trial.hyperparameters.get("num_train_epochs", 4)

# Get best dropout value
best_dropout = best_trial.hyperparameters.get("dropout", 0.1)

# Create final model with best dropout
final_model = MultiTaskBERT(MODEL_NAME, NUM_LABELS, dropout=best_dropout).to(device)

# Create final trainer
final_trainer = MultiTaskTrainer(
    model=final_model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=dev_ds,
    data_collator=collator,
    compute_metrics=compute_metrics
)

print("\n Training final model...")
final_trainer.train()

# FINAL EVALUATION

print("\n" + "="*80)
print("FINAL EVALUATION")
print("="*80)

eval_results = final_trainer.evaluate()

print("\n DEV SET RESULTS:")
print(f"  MAE Emotion: {eval_results['eval_mae_emotion']:.4f}")
print(f"  MAE Empathy: {eval_results['eval_mae_empathy']:.4f}")
print(f"  Accuracy Polarity: {eval_results['eval_accuracy_polarity']:.4f}")
print(f"  F1 Polarity: {eval_results['eval_f1_polarity']:.4f}")

# Save model
final_trainer.save_model("./best_model_deberta")
tokenizer.save_pretrained("./best_model_deberta")
print("\n Model saved to ./best_model_deberta")

# GENERATE TEST PREDICTIONS
print("\n" + "="*80)
print("GENERATING TEST PREDICTIONS")
print("="*80)

test_predictions = final_trainer.predict(test_ds)

pred_emotion = test_predictions.predictions[0]
pred_empathy = test_predictions.predictions[1]
pred_polarity_logits = test_predictions.predictions[2]
pred_polarity = np.argmax(pred_polarity_logits, axis=1)

# Convert polarity back to original range [1, 2, 3]
pred_polarity_original = pred_polarity + 1

submission_df = pd.DataFrame({
    'id': test_df['id'].values,
    'Emotion': pred_emotion,
    'EmotionalPolarity': pred_polarity_original,
    'Empathy': pred_empathy
})

submission_df.to_csv('predictions_bert.csv', index=False)
print(f"Predictions saved to predictions_bert.csv")
print(f"\n Submission shape: {submission_df.shape}")
print("\n First 5 predictions:")
print(submission_df.head())

print("\n" + "="*80)
print("ALL TASKS COMPLETED!")
print("="*80)
print(f"\n Best Hyperparameters:")
for key, value in best_trial.hyperparameters.items():
    print(f"  {key}: {value}")
print(f"\nFinal Dev Accuracy: {eval_results['eval_accuracy_polarity']:.4f}")
print("="*80)

✓ Imports successful!
PyTorch: 2.8.0+cu126
CUDA: True
Device: Tesla T4

DATA LOADING & PREPROCESSING
✓ Train: (7338, 5), Dev: (564, 5), Test: (1941, 2)
✓ Polarity: Train {0: 4398, 1: 2939, 2: 1}
✓ Polarity: Dev {0: 392, 1: 172}

MODEL SETUP
Model: microsoft/deberta-v3-base
Max length: 128
Num polarity classes: 3


/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


✓ Tokenizer loaded: DebertaV2TokenizerFast

TOKENIZATION


Map:   0%|          | 0/7338 [00:00<?, ? examples/s]

Map:   0%|          | 0/564 [00:00<?, ? examples/s]

...

[I 2025-11-02 11:29:49,759] Trial 9 finished with value: 0.8457446808510638 and parameters: {'learning_rate': 4.431851918199835e-05, 'per_device_train_batch_size': 16, 'num_train_epochs': 4, 'weight_decay': 0.005992992755281271, 'warmup_ratio': 0.14715811516437213, 'dropout': 0.1967499225701591}. Best is trial 3 with value: 0.8475177304964538.



HYPERPARAMETER SEARCH COMPLETE!

Best hyperparameters found:
  learning_rate: 1.6435226692536434e-05
  per_device_train_batch_size: 8
  num_train_epochs: 3
  weight_decay: 0.0413495649635518
  warmup_ratio: 0.10940889802305785
  dropout: 0.28570039873236563

Best accuracy: 0.8475

TRAINING FINAL MODEL WITH BEST HYPERPARAMETERS

Training final model...


Epoch,Training Loss,Validation Loss,Mae Emotion,Mae Empathy,Accuracy Polarity,F1 Polarity
1,1.420800,1.419118,0.471368,0.691114,0.865248,0.863309
2,1.215800,1.438682,0.507787,0.660968,0.863475,0.861375
3,1.068300,1.411997,0.472300,0.670536,0.867021,0.865494



FINAL EVALUATION



DEV SET RESULTS:
  MAE Emotion: 0.4723
  MAE Empathy: 0.6705
  Accuracy Polarity: 0.8670
  F1 Polarity: 0.8655

✓ Model saved to ./best_model_deberta

GENERATING TEST PREDICTIONS
✓ Predictions saved to predictions_bert.csv

Submission shape: (1941, 4)

First 5 predictions:
   id   Emotion  EmotionalPolarity   Empathy
0  27  2.126953                  1  1.738281
1  27  1.043945                  1  0.860840
2  27  1.070312                  1  0.888184
3  27  2.976562                  1  1.478516
4  27  2.251953                  2  2.343750

ALL TASKS COMPLETED!

Best Hyperparameters:
  learning_rate: 1.6435226692536434e-05
  per_device_train_batch_size: 8
  num_train_epochs: 3
  weight_decay: 0.0413495649635518
  warmup_ratio: 0.10940889802305785
  dropout: 0.28570039873236563

Final Dev Accuracy: 0.8670
Improvement over baseline (82.6%): +4.10%
